In [1]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.vectorstores import Chroma
from langchain_cohere import CohereRerank
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv


/home/orlandojunior/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-13 22:58:46.321514: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-13 22:58:47.197146: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-13 22:58:48.606717: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly differ

In [2]:
load_dotenv()

True

In [3]:
llm = ChatOpenAI()
embeddings = OpenAIEmbeddings()

In [4]:
pdf_link = "./os-sertoes.pdf"

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000,
    chunk_overlap=20,
    length_function=len,
    add_start_index=True
)

loader = PyPDFLoader(pdf_link,extract_images=False)
pages = loader.load_and_split()
len(pages)

658

In [5]:
chunks = text_splitter.split_documents(pages)
vectordb = Chroma(embedding_function=embeddings,persist_directory="chromadb3")

/tmp/ipykernel_145765/2111098954.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectordb = Chroma(embedding_function=embeddings,persist_directory="chromadb3")


In [6]:
vectordb.add_documents(chunks)
naive_retriever = vectordb.as_retriever(search_kwargs={"k":10})

In [7]:
rerank = CohereRerank(model="rerank-multilingual-v3.0",top_n=3)

compressor_retriver = ContextualCompressionRetriever(
    base_compressor=rerank,
    base_retriever=naive_retriever
)


In [8]:
TEMPLATE = """
    Você é um especialista no livro "os sertões. Responda a pergunta utilizando o contexto informado"
    Query:
    {question}

    Context:
    {context}
"""

rag_prompt = ChatPromptTemplate.from_template(TEMPLATE)

In [9]:
setup_retrival = RunnableParallel({"question": RunnablePassthrough(), "context":compressor_retriver})


In [10]:
compressor_retrieval_chain = setup_retrival | rag_prompt | llm | StrOutputParser()

In [11]:
questions = [
    "Qual é a visão de Euclides da Cunha sobre o ambiente natural do sertão nordestino e como ele influencia a vida dos habitantes?",
    "Quais são as principais características da população sertaneja descritas por Euclides da Cunha? Como ele relaciona essas características com o ambiente em que vivem?",
    "Qual foi o contexto histórico e político que levou à Guerra de Canudos, segundo Euclides da Cunha?",
    "Como Euclides da Cunha descreve a figura de Antônio Conselheiro e seu papel na Guerra de Canudos?",
    "Quais são os principais aspectos da crítica social e política presentes em 'Os Sertões'? Como esses aspectos refletem a visão do autor sobre o Brasil da época?"
]

for i,question in enumerate(questions):
    print(f'Questão {i} : {question}')
    print("Resposta:")
    print("\n")
    response = compressor_retrieval_chain.invoke(question)
    print(response)
    print("=============================")


Questão 0 : Qual é a visão de Euclides da Cunha sobre o ambiente natural do sertão nordestino e como ele influencia a vida dos habitantes?
Resposta:


Euclides da Cunha tinha uma visão complexa e profunda sobre o ambiente natural do sertão nordestino. Ele descrevia o sertão como um lugar de extrema adversidade, marcado por intermitências de cheias e estiagens, fenômenos climáticos extremos e desoladores. Essas condições naturais tornavam a vida dos habitantes do sertão ainda mais difícil, levando ao surgimento de anomalias e situações que impediam a continuidade de esforços e propiciavam o parasitismo. Euclides da Cunha percebia o sertão como um ambiente que impunha novas exigências biológicas e refletia regimes adversos, o que influenciava diretamente a vida e as atividades dos habitantes locais. Sua visão do sertão era marcada por uma mistura de admiração pela natureza e compaixão pela dura realidade enfrentada pelas pessoas que habitavam essa região.
Questão 1 : Quais são as princip